# 2.4 Position and Ranking

**Goal:** locate observations within a distribution and use quantiles and standardization consistently.

**Prerequisites:** central tendency, dispersion, and measurement scales. **Runtime:** Python 3.10+, standard library only. Synthetic data and saved outputs are included. Run cells in order.

## Contents
1. Quantiles and conventions
2. Quartiles
3. Deciles
4. Percentiles and percentile ranks
5. Five-number summary
6. Z-scores
7. Standardization
8. ML application, practice, and solutions

## 1. Quantiles

A population $p$-quantile is commonly defined as a generalized inverse of its CDF: $Q(p)=\inf\{x:F(x)\ge p\}$ for $0<p<1$. Informally, it marks a cumulative proportion $p$. Finite-sample quantile estimators need a convention: interpolation, nearest-rank rules, and order-statistic rules can disagree, especially in small datasets.

**Convention used throughout Module 2:** sort $x_{(0)}\le\cdots\le x_{(n-1)}$, compute $h=(n-1)p$, let $j=\lfloor h\rfloor$, and linearly interpolate between the surrounding observations. At endpoints use the minimum or maximum. This is often called **type 7**.

**Worked example:** for 10,20,30,40,50 at $p=0.30$, $h=4(0.30)=1.2$, giving $20+0.2(30-20)=22$. The estimate need not be an observed value. This interpolated sample quantile is not identical to the step-function inverse of the empirical CDF.

Avoid asserting that exactly 30% of the finite sample must fall below 22. Ties, discreteness, and the estimator convention prevent that simple equality.

In [1]:
import math
import statistics

def quantile(values, p):
    """Linear interpolation at position (n-1)*p (often called type 7)."""
    xs = sorted(values)
    if not xs or not 0 <= p <= 1:
        raise ValueError("Need nonempty values and p in [0, 1]")
    position = (len(xs)-1)*p
    low = math.floor(position)
    high = math.ceil(position)
    return xs[low] + (position-low)*(xs[high]-xs[low])

xs = [10, 20, 30, 40, 50]
print("30th percentile (linear interpolation):", quantile(xs, 0.30))
print("Fraction at or below 22:", sum(x <= 22 for x in xs)/len(xs))
assert quantile(xs, 0.30) == 22

30th percentile (linear interpolation): 22.0
Fraction at or below 22: 0.4


## 2. Quartiles

Quartiles divide a distribution into four conceptual parts: $Q_1=Q(0.25)$, $Q_2=Q(0.50)$ (median), and $Q_3=Q(0.75)$. IQR is $Q_3-Q_1$.

**Worked example:** for 10,20,30,40,50, quartiles are 20,30,40 and IQR is 20. Quantiles remain meaningful for ordered data, but arithmetic interpolation between ordinal category codes needs additional assumptions; the code here is for numerical data.

## 3. Deciles

Deciles mark tenths: $D_k=Q(k/10)$ for $k=1,\ldots,9$. For the example, $D_1=14$, $D_5=30$, and $D_9=46$. They offer more detail than quartiles but require enough observations for stable interpretation.

## 4. Percentiles

Percentiles mark hundredths: $P_k=Q(k/100)$. Thus $P_{25}=Q_1$, $P_{50}=Q_2$, and $P_{90}=D_9$ when the same convention is used.

A **percentile value** is a measurement at a cumulative position; a **percentile rank** is a relative position assigned to an observed value. These are different. A score of 90 is not automatically the 90th percentile, and the 90th percentile of response time is not 90% of the maximum response time.

One possible tie-aware rank convention is
$$100\frac{\#(x_i<x)+0.5\#(x_i=x)}n.$$
Other conventions use strictly-below or at-or-below counts. State the rule; ties make a difference. This midrank percentage is not an exact algebraic inverse of the interpolated quantile function.

In [2]:
print("Quartiles:", [quantile(xs, p) for p in [0.25, 0.5, 0.75]])
print("Deciles:", [quantile(xs, k/10) for k in range(1, 10)])
print("P90:", quantile(xs, 0.9))

def percentile_rank_midrank(values, value):
    if not values:
        raise ValueError("Need nonempty values")
    below = sum(x < value for x in values)
    equal = sum(x == value for x in values)
    return 100*(below + 0.5*equal)/len(values)

scores = [50, 60, 60, 80, 100]
print("Midrank percentile for 60:", percentile_rank_midrank(scores, 60))
print("Strictly-below percentage:", 100*sum(x < 60 for x in scores)/len(scores))
print("At-or-below percentage:", 100*sum(x <= 60 for x in scores)/len(scores))
assert percentile_rank_midrank(scores, 60) == 40

Quartiles: [20.0, 30.0, 40.0]
Deciles: [14.0, 18.0, 22.0, 26.0, 30.0, 34.0, 38.0, 42.0, 46.0]
P90: 46.0
Midrank percentile for 60: 40.0
Strictly-below percentage: 20.0
At-or-below percentage: 60.0


## 5. Five-Number Summary

The five-number summary is **minimum, $Q_1$, median, $Q_3$, maximum**. It captures endpoints, center, and central spread without listing every value. It does not reveal every peak, gap, or tail feature; many different datasets share the same summary.

**Worked example:** 10,20,30,40,50 have summary (10,20,30,40,50). Adding an extreme endpoint can dramatically change the maximum while leaving some central quantiles relatively stable.

A common Tukey box plot draws a box from $Q_1$ to $Q_3$, a median line, and whiskers to the most extreme **observed values within** the 1.5-IQR fences. Its whiskers are therefore not necessarily the minimum and maximum, and not necessarily the numerical fences. Other plotting conventions exist; specify the one used.

In [3]:
values = [1, 2, 3, 4, 5, 6, 7, 8, 30]
five_numbers = tuple(quantile(values, p) for p in [0, 0.25, 0.5, 0.75, 1])
q1, q3 = five_numbers[1], five_numbers[3]
iqr = q3-q1
inside = [x for x in values if q1-1.5*iqr <= x <= q3+1.5*iqr]
print("Five-number summary:", five_numbers)
print("Tukey whisker endpoints:", min(inside), max(inside))
assert five_numbers == (1, 3, 5, 7, 30)
assert max(inside) == 8

Five-number summary: (1, 3.0, 5.0, 7.0, 30)
Tukey whisker endpoints: 1 8


## 6. Z-scores

A population-standardized value is $z=(x-\mu)/\sigma$ for $\sigma>0$. A sample-based descriptive version uses $\bar x$ and $s$. In preprocessing, fitted training mean and SD are used instead; explicitly state the SD convention.

A z-score tells how many SD units a value is above (positive) or below (negative) its reference mean. It is dimensionless. It is not inherently a percentile or a probability.

**Worked example:** with reference mean 70 and SD 10, a score of 85 has $z=1.5$. Only if a normal distribution model is justified can a normal CDF translate it into an approximately 93.32nd percentile. For other distributions the percentile can differ greatly.

## 7. Standardization

Standardization commonly means transforming values using $z=(x-m)/s$, with $m,s$ estimated on training data. On the same nonconstant data used to fit the transform, the resulting mean is approximately 0 and SD is 1 **using the same denominator convention**. Held-out data need not have mean 0 or SD 1.

This is different from min-max scaling, $(x-x_{\min})/(x_{\max}-x_{\min})$, which maps training endpoints to 0 and 1. Standardization does not make a skewed distribution normal and does not cap values at ±1. Min-max transformed held-out values can lie outside [0,1].

Constant features have zero spread: choose a documented policy such as dropping the feature or mapping it to zero using a safe scale factor. Never divide blindly by zero.

In [4]:
print("Reference z-score:", (85-70)/10)
print("Normal-model cumulative probability:", statistics.NormalDist().cdf(1.5))
train = [10.0, 20.0, 30.0, 40.0]
mean = statistics.mean(train)
sd = statistics.pstdev(train)  # denominator n for descriptive training scaling
assert sd > 0
standardized = [(x-mean)/sd for x in train]
print("Training z-scores:", standardized)
print("Transformed mean/SD:", statistics.mean(standardized), statistics.pstdev(standardized))
held_out = [50.0, 60.0]
print("Held-out z-scores:", [(x-mean)/sd for x in held_out])
print("Held-out min-max values:", [(x-min(train))/(max(train)-min(train)) for x in held_out])
assert math.isclose(statistics.mean(standardized), 0, abs_tol=1e-12)
assert math.isclose(statistics.pstdev(standardized), 1)

Reference z-score: 1.5
Normal-model cumulative probability: 0.9331927987311419
Training z-scores: [-1.3416407864998738, -0.4472135954999579, 0.4472135954999579, 1.3416407864998738]
Transformed mean/SD: 0.0 1.0
Held-out z-scores: [2.23606797749979, 3.1304951684997055]
Held-out min-max values: [1.3333333333333333, 1.6666666666666667]


## 8. AI/ML application: rankings and scaling

Percentiles help summarize tail latency or choose a review threshold, but an empirical 95th-percentile cutoff is not a guaranteed future false-positive rate. Fit thresholds on appropriate training/calibration data and validate them independently. Distribution changes can invalidate earlier thresholds.

Z-score scaling can prevent large-unit features from dominating distance calculations or gradient updates. Choose preprocessing by model and data: tree splits often do not need the same scaling as distance-based methods. Outliers can inflate fitted SD, while IQR-based scaling offers a more robust alternative.

When filling missing values, estimate the imputation statistic on training data before fitting scaling. The order matters because imputation can alter means and variances.

## Practice

1. Using type-7 interpolation for 0,10,20,30,40, find $P_{25}$, $P_{60}$, and $D_9$.
2. Find the five-number summary of 1,2,3,4,5,6,7,8,30.
3. In scores 50,60,60,80,100, give the midrank percentile of 60 and the at-or-below percentage.
4. Find the z-score of 42 relative to mean 30 and SD 6.
5. Does z=2 always mean about 97.7% of observations lie below that value?
6. Why must validation values not affect the training scaling mean?
7. Does a standardized right-skewed variable become normal? Do held-out standardized values have to lie between -1 and 1?
8. What is the difference between the upper 1.5-IQR fence and a Tukey box-plot upper whisker?

## Solutions

1. Positions 1,2.4,3.6 give 10,24,36.
2. (1,3,5,7,30), with IQR 4.
3. Midrank: $100(1+0.5\times2)/5=40\%$; at-or-below: $100(3/5)=60\%$.
4. $(42-30)/6=2$.
5. No. That percentile interpretation assumes a normal model; a z-score itself is just a standardized distance.
6. It would leak held-out information into the fitted preprocessing and compromise an honest evaluation.
7. No to both. An affine transform preserves distribution shape up to scale/orientation and does not bound the transformed values.
8. The fence is the numerical threshold $Q_3+1.5\operatorname{IQR}$; the whisker reaches the largest observed value inside that fence.

**Readiness check:** state your quantile estimator and tie rule, distinguish percentile from rank, and keep fitted preprocessing independent of evaluation data.

**Next:** [2.5 Shape of Distribution](5_Shape-of-Distribution.ipynb).